In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'Pi_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.01
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = False
_MC_INCLUDE_BY_PREDICTOR = True
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: OutGap
Augmented coefficient: Pi_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[0.121 0.    0.   ]
 [0.    0.168 0.   ]
 [0.    0.    0.008]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
mc_reference["lb_summary"].round(3)

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,15.748,0.004,0.210,0.001,1000,983,0.983,0.004,0.973,0.989
1,Infl,23.226,0.000,0.245,0.000,1000,999,0.999,0.001,0.994,1.000
2,Rate,5.815,0.103,0.136,0.006,1000,606,0.606,0.015,0.575,0.636


In [8]:
display(mc_reference["moment_specification_test_summary"].round(3))

,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.123,2.698,0.552,0.002,0.079,0.009,1000,41,0.041,0.006,0.030,0.055,3.0,200,4
1,cov_identity,14.337,596.151,0.000,0.053,7.002,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-3.091,-0.118,1.857,-1.681,0.199,0.018,0.056,0.002,0.006,0.030,0.008,0.001,1000,390,0.390,0.015,0.360,0.421
1,OutGap,x,-0.174,-0.051,0.236,-0.721,0.436,0.007,0.007,0.002,0.001,0.030,0.009,0.000,1000,95,0.095,0.009,0.078,0.115
2,OutGap,r,-0.426,-0.013,2.035,-0.186,0.506,0.005,0.063,0.002,0.008,0.031,0.009,0.000,1000,51,0.051,0.007,0.039,0.066
3,Infl,Pi,-0.446,-0.013,2.145,-0.180,0.498,0.005,0.070,0.002,0.007,0.032,0.009,0.000,1000,63,0.063,0.008,0.050,0.080
4,Infl,x,0.026,0.007,0.272,0.092,0.503,0.005,0.009,0.002,0.001,0.032,0.009,0.000,1000,63,0.063,0.008,0.050,0.080
5,Infl,r,-0.350,-0.009,2.335,-0.126,0.485,0.005,0.077,0.002,0.009,0.033,0.009,0.000,1000,56,0.056,0.007,0.043,0.072
6,Rate,Pi,-0.015,-0.004,0.398,-0.055,0.504,0.005,0.013,0.002,0.001,0.032,0.009,0.000,1000,52,0.052,0.007,0.040,0.068
7,Rate,x,0.009,0.013,0.050,0.180,0.501,0.005,0.002,0.002,0.000,0.032,0.009,0.000,1000,56,0.056,0.007,0.043,0.072
8,Rate,r,-0.158,-0.024,0.434,-0.347,0.478,0.006,0.014,0.002,0.002,0.032,0.009,0.000,1000,62,0.062,0.008,0.049,0.079


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-4.184,-0.338,0.821,-5.080,0.000,0.117,0.026,0.002,0.001,0.029,0.000,0.001,1000,999,0.999,0.001,0.994,1.000
1,OutGap,x,-0.489,-0.319,0.102,-4.759,0.000,0.105,0.003,0.002,0.000,0.029,0.000,0.001,1000,999,0.999,0.001,0.994,1.000
0,OutGap,r,1.460,0.053,1.919,0.751,0.465,0.005,0.044,0.002,0.007,0.022,0.009,0.000,1000,40,0.040,0.006,0.030,0.054
5,Infl,Pi,-0.094,-0.005,1.001,-0.074,0.497,0.005,0.032,0.002,0.001,0.032,0.009,0.000,1000,47,0.047,0.007,0.036,0.062
4,Infl,x,-0.005,-0.001,0.124,-0.018,0.504,0.005,0.004,0.002,0.000,0.032,0.009,0.000,1000,49,0.049,0.007,0.037,0.064
3,Infl,r,-0.226,-0.006,2.203,-0.090,0.490,0.005,0.072,0.002,0.008,0.033,0.009,0.000,1000,60,0.060,0.008,0.047,0.076
8,Rate,Pi,0.037,0.013,0.186,0.183,0.508,0.005,0.006,0.002,0.000,0.031,0.009,0.000,1000,48,0.048,0.007,0.036,0.063
7,Rate,x,0.007,0.021,0.023,0.292,0.492,0.005,0.001,0.002,0.000,0.032,0.009,0.000,1000,62,0.062,0.008,0.049,0.079
6,Rate,r,-0.179,-0.029,0.409,-0.411,0.478,0.006,0.013,0.002,0.001,0.032,0.010,0.000,1000,73,0.073,0.008,0.058,0.091


In [11]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.786,-4.877,-3.091,-3.091,0.0,0.0,0.0,0.036,0.026,0.056,0.056,0.0,0.0,0.0
1,OutGap,x,0.012,-0.186,-0.174,-0.174,-0.0,0.0,0.0,0.004,0.003,0.007,0.007,0.0,0.0,0.0
2,OutGap,r,-0.153,-0.273,-0.426,-0.426,0.0,0.0,0.0,0.039,0.029,0.063,0.063,0.0,0.0,0.0
3,Infl,Pi,-0.014,-0.432,-0.446,-0.446,-0.0,0.0,0.0,0.007,0.069,0.070,0.070,0.0,0.0,0.0
4,Infl,x,0.001,0.026,0.026,0.026,-0.0,0.0,0.0,0.001,0.009,0.009,0.009,0.0,0.0,0.0
5,Infl,r,-0.007,-0.342,-0.350,-0.350,-0.0,0.0,0.0,0.008,0.077,0.077,0.077,0.0,0.0,0.0
6,Rate,Pi,0.002,-0.017,-0.015,-0.015,0.0,0.0,0.0,0.002,0.013,0.013,0.013,0.0,0.0,0.0
7,Rate,x,-0.000,0.009,0.009,0.009,-0.0,0.0,0.0,0.000,0.002,0.002,0.002,0.0,0.0,0.0
8,Rate,r,-0.002,-0.156,-0.158,-0.158,0.0,0.0,0.0,0.002,0.014,0.014,0.014,0.0,0.0,0.0


In [12]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.951,-6.135,-4.184,-4.184,0.0,0.0,0.0,0.016,0.011,0.026,0.026,0.0,0.0,0.0
1,OutGap,x,0.214,-0.703,-0.489,-0.489,-0.0,0.0,0.0,0.002,0.002,0.003,0.003,0.0,0.0,0.0
2,OutGap,r,-0.780,2.240,1.460,1.460,-0.0,0.0,0.0,0.048,0.035,0.044,0.044,0.0,0.0,0.0
3,Infl,Pi,-0.007,-0.088,-0.094,-0.094,-0.0,0.0,0.0,0.003,0.032,0.032,0.032,0.0,0.0,0.0
4,Infl,x,-0.001,-0.004,-0.005,-0.005,-0.0,0.0,0.0,0.000,0.004,0.004,0.004,0.0,0.0,0.0
5,Infl,r,-0.005,-0.221,-0.226,-0.226,0.0,0.0,0.0,0.007,0.073,0.072,0.072,0.0,0.0,0.0
6,Rate,Pi,0.002,0.035,0.037,0.037,0.0,0.0,0.0,0.001,0.006,0.006,0.006,0.0,0.0,0.0
7,Rate,x,0.000,0.007,0.007,0.007,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.002,-0.177,-0.179,-0.179,0.0,0.0,0.0,0.002,0.013,0.013,0.013,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [13]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model


### Marginal LR Test Conditional on $\theta_0$


In [15]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.962,-2193.2,-918.895,2548.611,0.0,0.002,5.34,0.832,10.093,0.0,1000,1000,1.0,0.0,0.996,1.0


In [16]:
res_mle

OptimizationResult(kind='mle', x=array([1.8637587]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(1.8637586971128686), 'x_coef': np.float64(0.0), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(872.4374334742786), loglik=np.float64(-872.4374334742786), logprior=np.float64(0.0), logpost=np.float64(-872.4374334742786), nfev=8, nit=3, raw=  message: CONVERGENCE: REL

## Serial Autocorrelation Tests for the Augmented Model


In [17]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.075,0.496,0.050,0.009,1000,60,0.060,0.008,0.047,0.076
1,Infl,12.105,0.012,0.188,0.001,1000,943,0.943,0.007,0.927,0.956
2,Rate,2.184,0.346,0.080,0.010,1000,199,0.199,0.013,0.175,0.225


In [18]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.123,2.698,0.552,0.002,0.079,0.009,1000,41,0.041,0.006,0.030,0.055,3.0,200,4
1,cov_identity,14.337,596.151,0.000,0.053,7.002,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.116,3.182,0.487,0.002,0.085,0.009,1000,69,0.069,0.008,0.055,0.086,3.0,200,4
1,cov_identity,0.992,32.034,0.002,0.007,0.315,0.000,1000,994,0.994,0.002,0.987,0.997,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.123,0.002,0.116,0.002,0.008,0.001,2.698,0.079,3.182,0.085,-0.484,0.034,0.607,0.015,0.576,0.637
1,cov_identity,1000,14.337,0.053,0.992,0.007,13.345,0.053,596.151,7.002,32.034,0.315,564.117,7.004,1.000,0.000,0.996,1.000
